# CAISO LMP download and processing

UPscaledEV / TotalEnergies project. This notebook downloads the price inputs actually used by the MPC: day-ahead hourly LMP and real-time market 5-minute LMP at node `UCM_6_N001`. It creates model-ready daily files and full-year 15-minute clean files.

Important conventions:
- OASIS raw LMP is in $/MWh; clean model prices are in $/kWh.
- Each query follows a California local operating day rather than a fixed UTC boundary.
- Spring/fall daylight-saving days are normalized to the model's fixed 24-hour clock. Missing spring intervals are time-interpolated; repeated fall intervals are averaged.
- Correct existing daily files are reused. Only missing or incorrectly aligned dates are downloaded.
- RTPD is not downloaded here because the current MPC settles real time with RTM prices.

Author: Yizhan Gu (UCSD CER)


In [ ]:
# Configuration and imports
import io
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

YEAR = 2025
START_DATE = pd.Timestamp(f"{YEAR}-01-01")
END_DATE = pd.Timestamp(f"{YEAR}-12-31")
NODE_NAME = "UCM_6_N001"
LOCAL_TIMEZONE = "America/Los_Angeles"
OASIS_URL = "https://oasis.caiso.com/oasisapi/SingleZip"
FORCE_REDOWNLOAD = False
REQUEST_PAUSE_SECONDS = 0.25
MAX_RETRIES = 5

PROJECT_ROOT = Path("/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024")
LMP_ROOT = PROJECT_ROOT / "2025Data" / "LMP" / str(YEAR)
LMP_ROOT.mkdir(parents=True, exist_ok=True)

MARKETS = {
    "DA": {
        "queryname": "PRC_LMP", "version": 12, "market_run_id": "DAM",
        "frequency": "1h", "value_column": "MW", "rows_per_day": 24,
    },
    "FM": {
        "queryname": "PRC_INTVL_LMP", "version": 3, "market_run_id": "RTM",
        "frequency": "5min", "value_column": "VALUE", "rows_per_day": 288,
    },
}

for market in MARKETS:
    (LMP_ROOT / market).mkdir(parents=True, exist_ok=True)
    (LMP_ROOT / "raw_oasis" / market).mkdir(parents=True, exist_ok=True)

print(f"Target: {START_DATE.date()} through {END_DATE.date()} at {NODE_NAME}")


In [ ]:
# Download, validation, and daylight-saving helpers
def local_day_utc_bounds(day):
    day = pd.Timestamp(day).normalize()
    start_local = day.tz_localize(LOCAL_TIMEZONE)
    end_local = (day + pd.DateOffset(days=1)).tz_localize(LOCAL_TIMEZONE)
    return start_local.tz_convert("UTC"), end_local.tz_convert("UTC")


def split_oasis_window(start_utc, end_utc):
    """CAISO OASIS requests are limited to at most 24 hours."""
    pieces = []
    cursor = start_utc
    while cursor < end_utc:
        next_cursor = min(cursor + pd.Timedelta(hours=24), end_utc)
        pieces.append((cursor, next_cursor))
        cursor = next_cursor
    return pieces


def oasis_time(ts):
    return pd.Timestamp(ts).tz_convert("UTC").strftime("%Y%m%dT%H:%M-0000")


def request_oasis_csv(params):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(OASIS_URL, params=params, timeout=90)
            response.raise_for_status()
            payload = io.BytesIO(response.content)
            if not zipfile.is_zipfile(payload):
                message = response.content[:800].decode("utf-8", errors="replace")
                raise RuntimeError(f"OASIS did not return a ZIP file: {message}")
            payload.seek(0)
            with zipfile.ZipFile(payload) as archive:
                csv_names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
                if not csv_names:
                    raise RuntimeError(f"No CSV found in OASIS response: {archive.namelist()}")
                frames = [pd.read_csv(archive.open(name)) for name in csv_names]
            return pd.concat(frames, ignore_index=True)
        except Exception as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(min(2 ** (attempt - 1), 12))
    raise RuntimeError(f"OASIS request failed after {MAX_RETRIES} attempts") from last_error


def expected_physical_rows(day, frequency):
    start_utc, end_utc = local_day_utc_bounds(day)
    minutes = int((end_utc - start_utc) / pd.Timedelta(minutes=1))
    step_minutes = int(pd.Timedelta(frequency) / pd.Timedelta(minutes=1))
    return minutes // step_minutes


def read_existing_day(day, market):
    """Return a valid existing daily LMP file, or None if it is missing/misaligned."""
    cfg = MARKETS[market]
    path = LMP_ROOT / market / f"{pd.Timestamp(day):%Y%m%d}_LMP.csv"
    if FORCE_REDOWNLOAD or not path.exists():
        return None, None
    try:
        frame = pd.read_csv(path)
        frame = frame[frame["LMP_TYPE"] == "LMP"].copy()
        if "LOCAL_DATETIME" in frame.columns:
            frame["_local"] = pd.to_datetime(frame["LOCAL_DATETIME"], errors="raise")
            if len(frame) == cfg["rows_per_day"] and frame["_local"].dt.date.nunique() == 1:
                return frame, "model_ready"
            return None, None
        utc = pd.to_datetime(frame["INTERVALSTARTTIME_GMT"], utc=True, errors="raise")
        frame["_utc"] = utc
        frame["_local"] = utc.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)
        frame = frame[frame["_local"].dt.date == pd.Timestamp(day).date()].copy()
        if frame["_utc"].nunique() != expected_physical_rows(day, cfg["frequency"]):
            return None, None
        return frame, "existing"
    except Exception:
        return None, None


def download_lmp_day(day, market):
    cfg = MARKETS[market]
    start_utc, end_utc = local_day_utc_bounds(day)
    chunks = []
    for chunk_start, chunk_end in split_oasis_window(start_utc, end_utc):
        params = {
            "resultformat": 6, "queryname": cfg["queryname"], "version": cfg["version"],
            "startdatetime": oasis_time(chunk_start), "enddatetime": oasis_time(chunk_end),
            "market_run_id": cfg["market_run_id"], "node": NODE_NAME,
        }
        chunks.append(request_oasis_csv(params))
        time.sleep(REQUEST_PAUSE_SECONDS)
    raw = pd.concat(chunks, ignore_index=True)
    raw = raw.drop_duplicates().reset_index(drop=True)
    raw_path = LMP_ROOT / "raw_oasis" / market / f"{pd.Timestamp(day):%Y%m%d}.csv"
    raw.to_csv(raw_path, index=False)
    frame = raw[raw["LMP_TYPE"] == "LMP"].copy()
    utc = pd.to_datetime(frame["INTERVALSTARTTIME_GMT"], utc=True, errors="raise")
    frame["_utc"] = utc
    frame["_local"] = utc.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)
    frame = frame[frame["_local"].dt.date == pd.Timestamp(day).date()].copy()
    expected = expected_physical_rows(day, cfg["frequency"])
    if frame["_utc"].nunique() != expected:
        raise ValueError(f"{day:%Y-%m-%d} {market}: expected {expected} physical rows, got {frame['_utc'].nunique()}")
    return frame


def normalize_lmp_day(frame, day, market):
    cfg = MARKETS[market]
    value_col = cfg["value_column"]
    frame[value_col] = pd.to_numeric(frame[value_col], errors="coerce")
    physical_rows = int(frame["_utc"].nunique()) if "_utc" in frame.columns else expected_physical_rows(day, cfg["frequency"])
    values = frame.groupby("_local")[value_col].mean().sort_index()
    target = pd.date_range(pd.Timestamp(day).normalize(), periods=cfg["rows_per_day"], freq=cfg["frequency"])
    values = values.reindex(target).interpolate(method="time", limit_direction="both")
    if values.isna().any():
        raise ValueError(f"{day:%Y-%m-%d} {market}: NaN remains after DST normalization")
    if physical_rows < cfg["rows_per_day"]:
        dst_action = "spring_gap_interpolated"
    elif physical_rows > cfg["rows_per_day"]:
        dst_action = "fall_duplicate_averaged"
    else:
        dst_action = "none"
    ready = pd.DataFrame({
        "INTERVALSTARTTIME_GMT": target.strftime("%Y-%m-%dT%H:%M:%S"),
        "LOCAL_DATETIME": target.strftime("%Y-%m-%d %H:%M:%S"),
        "OPR_DT": target.strftime("%Y-%m-%d"),
        "OPR_HR": target.hour + 1,
        "OPR_INTERVAL": 0 if market == "DA" else target.minute // 5 + 1,
        "NODE": NODE_NAME,
        "MARKET_RUN_ID": cfg["market_run_id"],
        "LMP_TYPE": "LMP",
        value_col: values.to_numpy(),
    })
    ready_path = LMP_ROOT / market / f"{pd.Timestamp(day):%Y%m%d}_LMP.csv"
    ready.to_csv(ready_path, index=False)
    return ready, physical_rows, dst_action


In [ ]:
# Run the full-year missing-only download and build model-ready price files
dates = pd.date_range(START_DATE, END_DATE, freq="D")
qc_rows = []
da_15min_days = []
fm_15min_days = []

for day in tqdm(dates, desc="LMP operating days"):
    for market in MARKETS:
        frame, source = read_existing_day(day, market)
        if frame is None:
            frame = download_lmp_day(day, market)
            source = "oasis_download"
        ready, physical_rows, dst_action = normalize_lmp_day(frame, day, market)
        cfg = MARKETS[market]
        value = ready[cfg["value_column"]].astype(float)
        if market == "DA":
            price = np.repeat(value.to_numpy(), 4) * 0.001
            index = pd.date_range(day, periods=96, freq="15min")
            da_15min_days.append(pd.DataFrame({"Datetime": index, "Price($/kWh)": price}))
        else:
            local_index = pd.to_datetime(ready["LOCAL_DATETIME"])
            price = pd.Series(value.to_numpy(), index=local_index).resample("15min").mean() * 0.001
            index = pd.date_range(day, periods=96, freq="15min")
            price = price.reindex(index).interpolate(method="time", limit_direction="both")
            fm_15min_days.append(pd.DataFrame({"Datetime": index, "Price($/kWh)": price.to_numpy()}))
        qc_rows.append({
            "date": day.strftime("%Y-%m-%d"), "market": market, "source": source,
            "physical_rows": physical_rows, "model_rows": len(ready),
            "dst_action": dst_action, "missing_values": int(value.isna().sum()), "purpose": "study_year",
        })

# A 24-hour Rolling MPC on Dec 31 also needs Jan 1 of the following year.
boundary_day = END_DATE + pd.DateOffset(days=1)
for market in MARKETS:
    frame, source = read_existing_day(boundary_day, market)
    if frame is None:
        frame = download_lmp_day(boundary_day, market)
        source = "oasis_download"
    ready, physical_rows, dst_action = normalize_lmp_day(frame, boundary_day, market)
    qc_rows.append({
        "date": boundary_day.strftime("%Y-%m-%d"), "market": market, "source": source,
        "physical_rows": physical_rows, "model_rows": len(ready),
        "dst_action": dst_action, "missing_values": int(ready.isna().sum().sum()), "purpose": "rolling_boundary",
    })

LMP_DA = pd.concat(da_15min_days, ignore_index=True)
LMP_FM = pd.concat(fm_15min_days, ignore_index=True)
LMP_QC = pd.DataFrame(qc_rows)

LMP_DA.to_csv(LMP_ROOT / f"LMP_DA_{YEAR}_clean.csv", index=False)
LMP_FM.to_csv(LMP_ROOT / f"LMP_FM_{YEAR}_clean.csv", index=False)
LMP_QC.to_csv(LMP_ROOT / f"LMP_{YEAR}_download_qc.csv", index=False)

expected_15min = len(dates) * 96
for label, frame in [("DA", LMP_DA), ("FM", LMP_FM)]:
    timestamp = pd.to_datetime(frame["Datetime"], errors="raise")
    assert len(frame) == expected_15min
    assert timestamp.nunique() == expected_15min
    assert frame["Price($/kWh)"].notna().all()
    print(f"{label}: {len(frame):,} clean 15-minute prices, {timestamp.min()} to {timestamp.max()}")

print("Download sources:")
display(LMP_QC.groupby(["market", "source"]).size().rename("days").reset_index())
print("DST normalization:")
display(LMP_QC[LMP_QC["dst_action"] != "none"])


In [ ]:
# Summary statistics and 300-dpi diagnostic figure
stats = pd.DataFrame({
    "LMP_DA_$/kWh": LMP_DA["Price($/kWh)"].describe(),
    "LMP_RTM_$/kWh": LMP_FM["Price($/kWh)"].describe(),
})
stats.to_csv(LMP_ROOT / f"LMP_Price_Statistics_{YEAR}.csv")
display(stats)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(pd.to_datetime(LMP_DA["Datetime"]), LMP_DA["Price($/kWh)"], lw=0.55, color="#3B82A0")
axes[1].plot(pd.to_datetime(LMP_FM["Datetime"]), LMP_FM["Price($/kWh)"], lw=0.55, color="#D97757")
for ax, title in zip(axes, ["Day-ahead LMP", "Real-time market LMP"]):
    ax.set_ylabel("$/kWh")
    ax.set_title(title, fontweight="bold")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
axes[-1].set_xlabel(str(YEAR))
fig.tight_layout()
fig.savefig(LMP_ROOT / f"LMP_Price_TimeSeries_{YEAR}.png", dpi=300, bbox_inches="tight")
plt.show()
